# NexusAI Core-01 — Kaggle GPU Diagnostic (Package-based, Synthetic Data Only)

**Input required**: `nexusai_core01_kaggle_package.zip` attached as a Kaggle Dataset/Input.
The notebook detects the ZIP under `/kaggle/input/`, extracts it to `/kaggle/working/nexusai_core01/`, and runs diagnostics on the extracted package.

**This notebook does NOT train, does NOT download datasets, does NOT update model weights.**
Run All is safe. If package init fails, downstream cells stop with a clear error.

In [ ]:
# Cell 1: Environment check
import sys, os, torch, platform
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "n/a")

In [ ]:
# Cell 2: GPU identity + memory
HAS_CUDA = torch.cuda.is_available()
if HAS_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
    !nvidia-smi
else:
    print("No CUDA GPU detected — CPU fallback mode (GPU checks will SKIP).")

In [ ]:
# Cell 3: Package init — find ZIP, extract, wire imports
# Sets PACKAGE_OK=True only if everything succeeds; later cells assert it.
import os, sys, glob, zipfile, shutil
import torch

PACKAGE_OK = False
PACKAGE_ROOT = None
AI_CORE = None
TOKENIZER_PATH = None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    # --- locate the package ZIP anywhere under the input dir ---
    input_dirs = ["/kaggle/input"]
    if os.environ.get("NEXUSAI_INPUT_DIR"):   # local/test override
        input_dirs.insert(0, os.environ["NEXUSAI_INPUT_DIR"])
    zip_hits = []
    for d in input_dirs:
        if os.path.isdir(d):
            zip_hits += glob.glob(os.path.join(d, "**", "nexusai_core01_kaggle_package.zip"), recursive=True)
    print("zip candidates:", zip_hits)

    work_root = os.environ.get("NEXUSAI_WORK_DIR", "/kaggle/working/nexusai_core01")
    os.makedirs(work_root, exist_ok=True)

    if zip_hits:
        zpath = zip_hits[0]
        dest = os.path.join(work_root, "pkg")
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(dest)
        # zip contains top folder nexusai_core01_kaggle_package/
        inner = os.path.join(dest, "nexusai_core01_kaggle_package")
        PACKAGE_ROOT = inner if os.path.isdir(inner) else dest
        print("extracted to:", PACKAGE_ROOT)
    else:
        # fallback: already-extracted package dir under input
        dirs = []
        for d in input_dirs:
            if os.path.isdir(d):
                dirs += glob.glob(os.path.join(d, "**", "nexusai_core01_kaggle_package"), recursive=True)
                dirs += glob.glob(os.path.join(d, "**", "core01"), recursive=True)
        print("extracted-dir candidates:", dirs)
        for cand in dirs:
            root = cand if os.path.basename(cand) == "nexusai_core01_kaggle_package" else os.path.dirname(cand)
            if os.path.isdir(os.path.join(root, "models", "nexusai_core_01")):
                PACKAGE_ROOT = root
                break

    assert PACKAGE_ROOT, "Package not found: attach nexusai_core01_kaggle_package.zip as Kaggle Input."

    # locate ai-core root = dir containing core01/ and models/
    if os.path.isdir(os.path.join(PACKAGE_ROOT, "core01")):
        AI_CORE = PACKAGE_ROOT
    else:
        AI_CORE = os.path.dirname(PACKAGE_ROOT)
    assert os.path.isdir(os.path.join(AI_CORE, "core01")), f"core01 not under {AI_CORE}"

    # imports
    sys.path.insert(0, AI_CORE)
    sys.path.insert(0, os.path.join(AI_CORE, "models"))
    from core01.transformer import NexusAICore01
    from nexusai_core_01.config.model_config import NexusAICore01Config
    from core01.tokenizer_interface import create_default_tokenizer
    from core01.training import PretrainingEngine, resolve_amp_dtype

    # tokenizer path resolved from inside the extracted package
    tok_candidate = os.path.join(
        AI_CORE, "data/experimental/phase9_2/tokenizers/bpe_8k/sp_bpe_8k.model")
    assert os.path.exists(tok_candidate), f"tokenizer missing: {tok_candidate}"
    TOKENIZER_PATH = tok_candidate

    PACKAGE_OK = True
    print("PACKAGE INIT OK — root:", AI_CORE)
except Exception as e:
    print("PACKAGE INIT FAILED:", type(e).__name__, e)
    PACKAGE_OK = False

assert PACKAGE_OK, "Stopping: package initialization failed (see error above)."

In [ ]:
# Cell 4: Tokenizer + model init + param count
assert PACKAGE_OK
cfg = NexusAICore01Config(device=DEVICE, precision="fp16" if DEVICE=="cuda" else "fp32")
tok = create_default_tokenizer(cfg)
print("tokenizer vocab:", tok.get_vocab_size())
assert tok.get_vocab_size() == 8000

model = NexusAICore01(cfg).to(DEVICE)
n_params = model.count_parameters()
print("params:", n_params)
assert n_params == 8_842_560, f"unexpected param count {n_params}"
if DEVICE == "cuda":
    print("VRAM used MB:", round(torch.cuda.memory_allocated()/1e6, 1))

In [ ]:
# Cell 5: Synthetic forward + backward (FP32; NO weight update)
assert PACKAGE_OK
import torch.nn.utils as _nnutils
torch.manual_seed(0)
x = torch.randint(0, 8000, (8, 512), device=DEVICE)
y = torch.randint(0, 8000, (8, 512), device=DEVICE)
model.train()
loss = model(x, y)
print("FP32 loss:", loss.item())
assert loss.item() == loss.item(), "NaN loss!"
loss.backward()
_nnutils.clip_grad_norm_(model.parameters(), 1.0)
model.zero_grad(set_to_none=True)   # discard grads — weights untouched
print("forward+backward OK (no optimizer step, no weight update)")

In [ ]:
# Cell 6: AMP checks — FP16 path on CUDA, capability report otherwise
assert PACKAGE_OK
for p in ("fp32", "fp16", "bf16"):
    print(f"amp[{p}] on {DEVICE} ->", resolve_amp_dtype(p, DEVICE))

if DEVICE == "cuda":
    scaler = torch.amp.GradScaler("cuda")
    dummy_opt = torch.optim.SGD(model.parameters(), lr=0.0)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        loss_amp = model(x, y)
    print("FP16 autocast loss:", loss_amp.item(), "dtype:", loss_amp.dtype)
    scaler.scale(loss_amp).backward()
    scaler.unscale_(dummy_opt)
    scaler.step(dummy_opt); scaler.update()
    model.zero_grad(set_to_none=True)
    print("AMP FP16 + GradScaler step OK (lr=0, no real update)")
else:
    print("AMP GPU test SKIPPED — no CUDA")

In [ ]:
# Cell 7: Checkpoint save + reload round-trip (diagnostic only)
assert PACKAGE_OK
ckdir = os.path.join(work_root, "ckpt_diagnostic")
engine = PretrainingEngine(model, cfg, ckdir, tokenizer_path=TOKENIZER_PATH)
engine.current_step = 10
path = engine.save_checkpoint(epoch=0, train_loss=9.9, val_loss=9.9,
                              filename="diag.pt")
print("saved:", path, "size MB:", round(os.path.getsize(path)/1e6,1))

model2 = NexusAICore01(cfg).to(DEVICE)
engine2 = PretrainingEngine(model2, cfg, ckdir)
engine2.load_checkpoint("diag.pt")
same = all(torch.equal(a.cpu(), b.cpu()) for a,b in
           zip(model.state_dict().values(), model2.state_dict().values())
           if torch.is_tensor(a))
print("weights restored identical:", same, "| step:", engine2.current_step)
assert same, "checkpoint reload mismatch!"

In [ ]:
# Cell 8: Throughput micro-probe (10 fwd/bwd only, weights NOT updated)
assert PACKAGE_OK
import time
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
model.train()
torch.cuda.synchronize() if DEVICE=="cuda" else None
t0=time.time()
for i in range(10):
    loss = model(x, y)
    loss.backward()
    model.zero_grad(set_to_none=True)
if DEVICE == "cuda":
    torch.cuda.synchronize()
dt=time.time()-t0
tok_per_s = 10*8*512/dt
print(f"10 fwd/bwd in {dt:.1f}s -> ~{tok_per_s:,.0f} tokens/sec (batch8 seq512, NO optimizer step)")
if DEVICE == "cuda":
    print("VRAM peak MB:", round(torch.cuda.max_memory_allocated()/1e6,1))
print("NOTE: probe only — real training needs owner approval.")

In [ ]:
# Cell 9: PASS/FAIL summary
results = {
    "CUDA available": HAS_CUDA,
    "package init": PACKAGE_OK,
    "param count == 8,842,560": n_params == 8_842_560,
    "tokenizer vocab == 8000": tok.get_vocab_size() == 8000,
    "synthetic fwd/bwd": True,
    "checkpoint round-trip": same,
}
print("=" * 55)
all_pass = True
for k, v in results.items():
    tag = "PASS" if v else ("SKIP" if k == "CUDA available" and not v else "FAIL")
    if k == "CUDA available" and not v:
        v = True  # CPU mode is supported, not a failure
    print(f"  {tag}: {k}")
    all_pass &= bool(v)
print("=" * 55)
print("DIAGNOSTIC", "PASSED" if all_pass else "FAILED")
assert all_pass